# Imports

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

# Load Product Data

In [4]:
product_df = pd.read_csv(
    "../data/raw/olist_products_dataset.csv"
)

product_df.shape

(32951, 9)

# Select Features

In [5]:
product_features = (
    product_df[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ]
    .copy()
)

product_features.head()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0


# Handle Missing Values

In [6]:
product_features[
    "product_category_name"
] = (
    product_features[
        "product_category_name"
    ]
    .fillna("unknown")
)

In [7]:
numeric_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in numeric_cols:
    
    product_features[col] = (
        product_features[col]
        .fillna(
            product_features[col].median()
        )
    )

# Verify

In [8]:
product_features.isna().sum()

product_id               0
product_category_name    0
product_weight_g         0
product_length_cm        0
product_height_cm        0
product_width_cm         0
dtype: int64

# Save Clean Product Features

In [9]:
product_features.to_parquet(
    "../data/processed/product_features.parquet",
    index=False
)

# Verify Save

In [10]:
saved_product_features = pd.read_parquet(
    "../data/processed/product_features.parquet"
)

saved_product_features.shape

(32951, 6)

In [11]:
print(product_features.shape)

print(product_features.isna().sum())

(32951, 6)
product_id               0
product_category_name    0
product_weight_g         0
product_length_cm        0
product_height_cm        0
product_width_cm         0
dtype: int64


# Load Product Features

In [12]:
product_features = pd.read_parquet(
    "../data/processed/product_features.parquet"
)

product_features.head()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0


# One-Hot Encode Category

In [13]:
from sklearn.preprocessing import OneHotEncoder

In [14]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

category_encoded = encoder.fit_transform(
    product_features[
        ["product_category_name"]
    ]
)

category_encoded.shape

(32951, 74)

# Scale Numerical Features

In [16]:
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

scaler = StandardScaler()

numeric_scaled = scaler.fit_transform(
    product_features[numeric_cols]
)

numeric_scaled.shape

(32951, 4)

# Build Feature Matrix

In [17]:
import numpy as np


feature_matrix = np.hstack(
    [
        category_encoded,
        numeric_scaled
    ]
)

feature_matrix.shape

(32951, 78)

# Save Feature Matrix

In [18]:
np.save(
    "../artifacts/product_feature_matrix.npy",
    feature_matrix
)

In [19]:
product_features[
    ["product_id"]
].to_parquet(
    "../artifacts/product_index_mapping.parquet",
    index=False
)

# Similarity Calculation

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
similarity_matrix = cosine_similarity(
    feature_matrix[:1000]
)

In [22]:
similarity_matrix.shape

(1000, 1000)

In [23]:
print(category_encoded.shape)
print(numeric_scaled.shape)
print(feature_matrix.shape)

(32951, 74)
(32951, 4)
(32951, 78)


# Build Nearest Neighbor Model

In [24]:
from sklearn.neighbors import NearestNeighbors

In [25]:
nn_model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine"
)

nn_model.fit(feature_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [26]:
import joblib

joblib.dump(
    nn_model,
    "../artifacts/content_based_nn_model.pkl"
)

['../artifacts/content_based_nn_model.pkl']

In [27]:
product_id_to_index = {
    product_id: idx
    for idx, product_id in enumerate(
        product_features["product_id"]
    )
}

In [28]:
index_to_product_id = {
    idx: product_id
    for idx, product_id in enumerate(
        product_features["product_id"]
    )
}

# Recommendation Function

In [29]:
def get_similar_products(
    product_id,
    top_k=10
):
    
    product_idx = (
        product_id_to_index[
            product_id
        ]
    )
    
    distances, indices = (
        nn_model.kneighbors(
            feature_matrix[
                product_idx
            ].reshape(1, -1)
        )
    )
    
    similar_products = [
        index_to_product_id[idx]
        for idx in indices[0][1:]
    ]
    
    return similar_products

# Test Recommender

In [32]:
sample_product = (
    product_features["product_id"]
    .iloc[0]
)

sample_product

'1e9e8ef04dbcff4541ed26657ea517e5'

In [33]:
similar_products = get_similar_products(
    sample_product,
    top_k=10
)

similar_products

['6cd51331a07b84149502aa6c9c5b536e',
 'ddc01db3147c02cb8f547ac87da1718f',
 '7198684d5c66560fe6d2061944263991',
 'eea3e07f864a0a1389726d8a5f31c3f6',
 '7634da152a4610f1595efa32f14722fc',
 '2007f2b457d10773f0d40622a433a22c',
 '5a2f62420bd55b28cdc512cd79263c73',
 '57eea82fdaafa70e1bda51d338583f9b',
 '5ac31d4825e0f3d9077465c57d4cef3b',
 'd51e0a7f437c0d14f560082ed007fd85']

In [34]:
products_to_view = (
    [sample_product]
    +
    similar_products
)

product_features[
    product_features["product_id"]
    .isin(products_to_view)
][
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1237,d51e0a7f437c0d14f560082ed007fd85,perfumaria,210.0,17.0,11.0,14.0
3186,7198684d5c66560fe6d2061944263991,perfumaria,195.0,16.0,11.0,14.0
6157,2007f2b457d10773f0d40622a433a22c,perfumaria,100.0,16.0,11.0,14.0
10839,5a2f62420bd55b28cdc512cd79263c73,perfumaria,100.0,16.0,11.0,14.0
15403,7634da152a4610f1595efa32f14722fc,perfumaria,200.0,16.0,10.0,15.0
19400,57eea82fdaafa70e1bda51d338583f9b,perfumaria,100.0,16.0,11.0,14.0
20404,ddc01db3147c02cb8f547ac87da1718f,perfumaria,160.0,17.0,10.0,14.0
22844,6cd51331a07b84149502aa6c9c5b536e,perfumaria,183.0,16.0,10.0,14.0
24574,5ac31d4825e0f3d9077465c57d4cef3b,perfumaria,100.0,16.0,11.0,14.0


In [35]:
print(feature_matrix.shape)

print(len(product_id_to_index))

print(len(index_to_product_id))

(32951, 78)
32951
32951


In [36]:
products_to_view = (
    [sample_product]
    +
    similar_products
)

product_features[
    product_features["product_id"]
    .isin(products_to_view)
][
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].sort_values(
    "product_category_name"
)

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1237,d51e0a7f437c0d14f560082ed007fd85,perfumaria,210.0,17.0,11.0,14.0
3186,7198684d5c66560fe6d2061944263991,perfumaria,195.0,16.0,11.0,14.0
6157,2007f2b457d10773f0d40622a433a22c,perfumaria,100.0,16.0,11.0,14.0
10839,5a2f62420bd55b28cdc512cd79263c73,perfumaria,100.0,16.0,11.0,14.0
15403,7634da152a4610f1595efa32f14722fc,perfumaria,200.0,16.0,10.0,15.0
19400,57eea82fdaafa70e1bda51d338583f9b,perfumaria,100.0,16.0,11.0,14.0
20404,ddc01db3147c02cb8f547ac87da1718f,perfumaria,160.0,17.0,10.0,14.0
22844,6cd51331a07b84149502aa6c9c5b536e,perfumaria,183.0,16.0,10.0,14.0
24574,5ac31d4825e0f3d9077465c57d4cef3b,perfumaria,100.0,16.0,11.0,14.0


In [37]:
import joblib

joblib.dump(
    product_id_to_index,
    "../artifacts/product_id_to_index.pkl"
)

joblib.dump(
    index_to_product_id,
    "../artifacts/index_to_product_id.pkl"
)

['../artifacts/index_to_product_id.pkl']

In [38]:
import json

model_3_metrics = {
    "model_type": "content_based",
    "products": int(len(product_features)),
    "features": int(feature_matrix.shape[1]),
    "neighbors": 10
}

with open(
    "../artifacts/model_3_metrics.json",
    "w"
) as f:
    json.dump(
        model_3_metrics,
        f,
        indent=4
    )

In [39]:
product_features.to_parquet(
    "../artifacts/product_features.parquet",
    index=False
)

In [40]:
from pathlib import Path

for file in Path("../artifacts").iterdir():
    print(file.name)

category_popularity.parquet
content_based_nn_model.pkl
customer_favorite_category.parquet
index_to_product_id.pkl
model_1_metrics.json
model_2_metrics.json
model_3_metrics.json
popularity_products.parquet
popularity_recommendations.parquet
popularity_recommender.parquet
product_features.parquet
product_feature_matrix.npy
product_id_to_index.pkl
product_index_mapping.parquet
